# Лабораторна робота №2. Частина 2
## Наука про дані: підготовчий етап
### Імпорт модулів та завантаження й очищення датасету

In [3]:
import os
import timeit
import urllib.request
import zipfile
import numpy as np
import pandas as pd

# Назва потрібного файлу
DATASET_PATH = "household_power_consumption.txt"
ZIP_URL = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
ZIP_PATH = "household_power_consumption.zip"

def download_and_extract_dataset():
    # Якщо розпакований файл уже є, нічого не робимо
    if os.path.exists(DATASET_PATH):
        print(f"Файл {DATASET_PATH} вже є у папці. Скачування не потрібне.")
        return

    print("Файл датасету відсутній. Починаємо автоматичне завантаження...")
    try:
        # Завантажуємо zip-архів з репозиторію UCI
        urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
        print("Архів успішно завантажено. Розпакування...")
        
        # Розпаковуємо текстовий файл поруч із ноутбуком
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(".")
        print("Розпакування завершено!")
        
        # Видаляємо тимчасовий zip-архів, щоб не займав місце
        os.remove(ZIP_PATH)
    except Exception as e:
        print(f"Помилка завантаження датасету: {e}")

def load_and_clean_data(filepath):
    # Примусово запускаємо завантаження, якщо файлу немає
    download_and_extract_dataset()
    
    print("Зчитування датасету (це може зайняти до 15-20 секунд)...")
    df = pd.read_csv(filepath, sep=';', low_memory=False)
    
    df = df.replace('?', np.nan)
    numeric_cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage', 
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    df = df.dropna(subset=numeric_cols)
    
    # ВИМОГА ВИКЛАДАЧА: Перетворення Date та Time у відповідний тип даних на самому початку
    df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
    df = df.dropna(subset=['DateTime'])
    df['Time_Parsed'] = df['DateTime'].dt.time
    
    return df

# Оцінка часу виконання завантаження та очищення даних
start_time = timeit.default_timer()
main_df = load_and_clean_data(DATASET_PATH)
elapsed_time = timeit.default_timer() - start_time

print(f"Загальна кількість записів після очищення: {len(main_df)}")
print(f"Час виконання процедури очищення: {elapsed_time:.4f} секунд")
display(main_df.head())

Файл датасету відсутній. Починаємо автоматичне завантаження...
Архів успішно завантажено. Розпакування...
Розпакування завершено!
Зчитування датасету (це може зайняти до 15-20 секунд)...
Загальна кількість записів після очищення: 2049280
Час виконання процедури очищення: 43.0390 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,Time_Parsed
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00,17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00,17:28:00


### Завдання 1:
Обрати всі записи, у яких загальна активна споживана потужність перевищує $5\text{ кВт}$.

In [4]:
def select_power_exceeding_5(df):
    return df[df['Global_active_power'] > 5]

start_time = timeit.default_timer()
result_task1 = select_power_exceeding_5(main_df)
elapsed_time = timeit.default_timer() - start_time

print(f"Кількість знайдених записів: {len(result_task1)}")
print(f"Час виконання процедури: {elapsed_time:.4f} секунд")
if not result_task1.empty:
    display(result_task1.head())

Кількість знайдених записів: 17547
Час виконання процедури: 0.0109 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,Time_Parsed
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,17:27:00
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00,17:35:00
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00,17:36:00


### Завдання 2:
Обрати всі записи, у яких сила струму лежить в межах $19\text{--}20\text{ А}$, для них виявити ті, у яких пральна машина та холодильник споживають більше, ніж бойлер та кондиціонер.

In [5]:
def select_intensity_and_submetering(df):
    # Фільтрація за силою струму в межах 19-20 А та порівняння груп споживання 2 і 3
    condition = (df['Global_intensity'] >= 19) & \
                (df['Global_intensity'] <= 20) & \
                (df['Sub_metering_2'] > df['Sub_metering_3'])
    return df[condition]

start_time = timeit.default_timer()
result_task2 = select_intensity_and_submetering(main_df)
elapsed_time = timeit.default_timer() - start_time

print(f"Кількість знайдених записів: {len(result_task2)}")
print(f"Час виконання процедури: {elapsed_time:.4f} секунд")
if not result_task2.empty:
    display(result_task2.head())

Кількість знайдених записів: 2509
Час виконання процедури: 0.0215 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,Time_Parsed
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00,18:09:00
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00,01:04:00
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00,01:08:00
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00,01:19:00
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00,01:20:00


### Завдання 3:
Обрати випадковим чином $500\text{ }000$ записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [ ]:
def random_sample_and_means(df, sample_size=500000):
    if len(df) < sample_size:
        sample_df = df.sample(n=len(df), replace=False)
    else:
        sample_df = df.sample(n=sample_size, replace=False)
        
    means = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    # Перетворюємо Series у DataFrame для ідеального табличного виведення через display
    means_df = means.to_frame(name='Середнє значення')
    return means_df

start_time = timeit.default_timer()
result_task3 = random_sample_and_means(main_df)
elapsed_time = timeit.default_timer() - start_time

print("Середні величини для 3-х груп споживання:")
# Використовуємо display() 
display(result_task3)
print(f"Час виконання процедури: {elapsed_time:.4f} секунд")

Середні величини для 3-х груп споживання:


,Середнє значення
Sub_metering_1,1.126762
Sub_metering_2,1.297456
Sub_metering_3,6.459822


Час виконання процедури: 0.3472 секунд


### Завдання 4:
Обрати ті записи, які після 18-00 споживають понад $6\text{ кВт}$ за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [ ]:
import datetime

def complex_selection_and_slicing(df):
    # Порівняння об'єктів часу замість рядків string
    time_threshold = datetime.time(18, 0, 0)
    
    condition = (df['Time_Parsed'] > time_threshold) & \
                (df['Global_active_power'] > 6) & \
                (df['Sub_metering_2'] > df['Sub_metering_1']) & \
                (df['Sub_metering_2'] > df['Sub_metering_3'])
    
    filtered_df = df[condition]
    
    if filtered_df.empty:
        return pd.DataFrame()
        
    half_index = len(filtered_df) // 2
    
    # Вибірка кожного третього елемента з першої половини
    first_half = filtered_df.iloc[:half_index:3]
    # Вибірка кожного четвертого елемента з другої половини
    second_half = filtered_df.iloc[half_index::4]
    
    return pd.concat([first_half, second_half])

start_time = timeit.default_timer()
result_task4 = complex_selection_and_slicing(main_df)
elapsed_time = timeit.default_timer() - start_time

print(f"Кількість отриманих записів після кроку зрізання: {len(result_task4)}")
print(f"Час виконання процедури: {elapsed_time:.4f} секунд")
if not result_task4.empty:
    display(result_task4.head())

Кількість отриманих записів після кроку зрізання: 310
Час виконання процедури: 0.0915 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,Time_Parsed
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00,18:05:00
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00,18:08:00
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00,20:58:00
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00,21:02:00
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00,21:05:00


### Завдання 5:
Пронормувати та стандартизувати вибраний датасет.

In [8]:
def normalize_and_standardize(df):
    numeric_cols = [
        'Global_active_power', 'Global_reactive_power', 'Voltage', 
        'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]
    
    df_norm = df.copy()
    df_std = df.copy()
    
    # Нормалізація (Min-Max Scaling)
    df_norm[numeric_cols] = (df[numeric_cols] - df[numeric_cols].min()) / (df[numeric_cols].max() - df[numeric_cols].min())
    
    # Стандартизація (Z-score Normalization)
    df_std[numeric_cols] = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std()
    
    return df_norm, df_std

start_time = timeit.default_timer()
normalized_df, standardized_df = normalize_and_standardize(main_df)
elapsed_time = timeit.default_timer() - start_time

print(f"Час виконання нормалізації та стандартизації: {elapsed_time:.4f} секунд")
print("Приклад нормованого датасету:")
display(normalized_df.head(2))
print("Приклад стандартизованого датасету:")
display(standardized_df.head(2))

Час виконання нормалізації та стандартизації: 1.8176 секунд
Приклад нормованого датасету:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,Time_Parsed
0,16/12/2006,17:24:00,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387,2006-12-16 17:24:00,17:24:00
1,16/12/2006,17:25:00,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129,2006-12-16 17:25:00,17:25:00


Приклад стандартизованого датасету:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,Time_Parsed
0,16/12/2006,17:24:00,2.955076,2.610720,-1.851816,3.098788,-0.182337,-0.051274,1.249420,2006-12-16 17:24:00,17:24:00
1,16/12/2006,17:25:00,4.037084,2.770405,-2.225274,4.133799,-0.182337,-0.051274,1.130897,2006-12-16 17:25:00,17:25:00


### Завдання 6:
Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [ ]:
def calculate_correlations(df, col1='Global_active_power', col2='Global_intensity'):
    pearson_corr = df[col1].corr(df[col2], method='pearson')
    spearman_corr = df[col1].corr(df[col2], method='spearman')
    return pearson_corr, spearman_corr

start_time = timeit.default_timer()
pearson, spearman = calculate_correlations(main_df)
elapsed_time = timeit.default_timer() - start_time

#  Перетворюємо результати на DataFrame для гарного виведення через display()
corr_df = pd.DataFrame({
    'Метод розрахунку': ['Пірсона (Pearson)', 'Спірмена (Spearman)'],
    'Коефіцієнт кореляції': [pearson, spearman]
})

print("Результати розрахунку коефіцієнтів кореляції:")
display(corr_df)
print(f"Час виконання процедури розрахунку кореляцій: {elapsed_time:.4f} секунд")

Результати розрахунку коефіцієнтів кореляції:


,Метод розрахунку,Коефіцієнт кореляції
0,Пірсона (Pearson),0.998889
1,Спірмена (Spearman),0.995372


Час виконання процедури розрахунку кореляцій: 6.6937 секунд


### Завдання 7:
Провести One Hot Encoding категоріального атрибута.

In [10]:
def perform_one_hot_encoding(df):
    # Створення штучного категоріального атрибуту на основі рівнів споживання
    df_with_cat = df.copy()
    df_with_cat['Power_Class'] = pd.cut(
        df_with_cat['Global_active_power'], 
        bins=[0, 1.5, 4.0, np.inf], 
        labels=['Low', 'Medium', 'High']
    )
    
    # Виконання One Hot Encoding для створеного атрибуту
    encoded_df = pd.get_dummies(df_with_cat, columns=['Power_Class'], dtype=int)
    return encoded_df

start_time = timeit.default_timer()
result_task7 = perform_one_hot_encoding(main_df)
elapsed_time = timeit.default_timer() - start_time

print(f"Час виконання процедури One Hot Encoding: {elapsed_time:.4f} секунд")
# ЗАМІНЕНО НА .sample(10) за вимогою викладача, щоб показати випадкові варіанти енкодінгу
display(result_task7.sample(10))

Час виконання процедури One Hot Encoding: 0.2194 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,Time_Parsed,Power_Class_Low,Power_Class_Medium,Power_Class_High
1519732,6/11/2009,02:16:00,0.340,0.114,245.44,1.4,0.0,0.0,0.0,2009-11-06 02:16:00,02:16:00,1,0,0
1261283,10/5/2009,14:47:00,1.284,0.348,241.39,5.6,1.0,3.0,12.0,2009-05-10 14:47:00,14:47:00,1,0,0
1780169,5/5/2010,22:53:00,0.964,0.244,238.71,4.0,0.0,2.0,1.0,2010-05-05 22:53:00,22:53:00,1,0,0
1892680,23/7/2010,02:04:00,0.540,0.204,236.94,2.4,0.0,1.0,1.0,2010-07-23 02:04:00,02:04:00,1,0,0
2026043,23/10/2010,16:47:00,1.568,0.064,243.97,6.4,0.0,3.0,19.0,2010-10-23 16:47:00,16:47:00,0,1,0
372040,1/9/2007,02:04:00,0.226,0.124,239.42,1.2,0.0,0.0,0.0,2007-09-01 02:04:00,02:04:00,1,0,0
1013074,19/11/2008,05:58:00,0.376,0.082,243.29,1.6,0.0,0.0,0.0,2008-11-19 05:58:00,05:58:00,1,0,0
392601,15/9/2007,08:45:00,1.854,0.118,239.55,7.6,0.0,0.0,18.0,2007-09-15 08:45:00,08:45:00,0,1,0
1143926,18/2/2009,02:50:00,0.310,0.058,244.30,1.2,0.0,0.0,0.0,2009-02-18 02:50:00,02:50:00,1,0,0
293383,8/7/2007,11:07:00,2.046,0.070,235.28,9.0,0.0,3.0,17.0,2007-07-08 11:07:00,11:07:00,0,1,0
